# Parameter Recovery

This notebook covers simulated parameter recovery of drift rate and theta according given avgWTP_left, avgWTP_right, and fixation. Fixations are generated based on empirical distributions following Tavares' method.

In [ ]:
from ast import literal_eval
from simulation import get_corrected_subject_empirical_distributions
import pandas as pd
import numpy as np
import simulation

df = pd.read_csv('/Users/braydenchien/Desktop/Enkavilab/DDM/1ms_trial_data.csv')
df['RT'] = df['RT']*1000 # adjustment for RT
df['fixation'] = df['fixation'].apply(literal_eval)

value_diffs = np.arange(-4, 4.25, 0.25)
legend = {
    "left": {1},
    "right": {2},
    "transition": {0}, 
    "blank_fixation": {4}
}
fixation_col = 'fixation'
left_value_col = 'avgWTP_left'
right_value_col = 'avgWTP_right'

empirical_distributions = get_corrected_subject_empirical_distributions(
    df, 
    value_diffs, 
    legend=legend, 
    fixation_col=fixation_col, 
    left_value_col=left_value_col, 
    right_value_col=right_value_col, 
    cutoff=0.95
)

Create and simulate trials given model conditions.

In [ ]:
from simulation import create_trials, simulate
dt = 0.001
seed = 42
model_conditions = {'drift_rate': 1.2, 'theta': 0.38, 'noise': 0.5}

trials = create_trials(500, empirical_distributions, seed=seed)

results_df = simulate(dt, model_conditions, trials, seed=seed, save_results=False)

## What do simulations look like?

To see how many trials in the sample for an rvd, run the following:

In [ ]:
import matplotlib.pyplot as plt

myarr = np.arange(-5, 5.25, 0.25)
mydict=dict.fromkeys(myarr, 0)

for index, trial in results_df.iterrows():
    mydict[trial['avgWTP_left'] - trial['avgWTP_right']] += 1

# unpack dictionary
keys = list(mydict.keys())
values = list(mydict.values())


plt.xlim(-5, 5)
plt.xticks(range(-5, 6, 1))
plt.bar(keys, values)
plt.xlabel("Signed RVD")
plt.ylabel("Number of Trials")
plt.show()

To inspect any trial from any simulation, call `plot_trajectory`. It takes the `results_df` from above and not a csv currently.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "svg.fonttype": "none",   # keep text as text
    "font.size": 22,
    "axes.titlesize": 26,
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
})

# Pale OPAQUE tints (no alpha needed)
LEFT_SHADE  = "#d9eaf7"  # light blue
RIGHT_SHADE = "#f9d7d7"  # light red

def plot_trajectory(dt, trial, model_conditions, save_path=None):
    """
    Plots the trajectory of the decision variable over time,
    shading periods of fixation left/right.

    Parameters:
    - trial: dict with keys:
        - 'trajectory': list or np.array of decision variable
        - 'fixation': list or np.array of fixation states (1=left, 2=right)
    - model_conditions: dict with keys 'drift_rate', 'theta', 'noise'
    - save_path: optional SVG path
    """
    # NOTE: assuming you have simulation.create_model in scope
    model = simulation.create_model( 
        model_conditions['drift_rate'],
        model_conditions['theta'],
        model_conditions['noise'],
        dt
    )

    trajectory = np.array(trial['trajectory'])
    fixation   = np.array(trial['fixation'])
    timesteps  = np.arange(len(trajectory)) * model.dt

    # --- Figure / axes with WHITE background to avoid darkening via transparency compositing ---
    fig, ax = plt.subplots(figsize=(15, 10))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    # Trajectory on top
    ax.plot(timesteps, trajectory, color="black", linewidth=2.0, label="Trajectory", zorder=3)

    # Identify contiguous fixation segments
    current_state = fixation[0]
    start_idx = 0
    for i in range(1, len(fixation)):
        is_state_change = fixation[i] != current_state
        is_last_point   = (i == len(fixation) - 1)
        if is_state_change or is_last_point:
            # include the last index in the span if we are at the very end
            end_idx = i if is_state_change else i + 1
            x0, x1 = timesteps[start_idx], timesteps[end_idx - 1]

            if current_state == 1:
                # OPAQUE pale fill, no edge, behind the line
                ax.axvspan(x0, x1, color=LEFT_SHADE, ec="none", zorder=1,
                           label="Fixation Left" if start_idx == 0 else None)
            elif current_state == 2:
                ax.axvspan(x0, x1, color=RIGHT_SHADE, ec="none", zorder=1,
                           label="Fixation Right" if start_idx == 0 else None)

            start_idx = i
            current_state = fixation[i]

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Decision Variable")
    ax.set_title("aDDM Accumulation Trajectory with Fixation Shading")
    ax.grid(True, zorder=0)

    # De-duplicate legend
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    # Drop any None keys that can sneak in when label=None above
    by_label = {k: v for k, v in by_label.items() if k is not None}
    ax.legend(by_label.values(), by_label.keys(), frameon=False)

    plt.tight_layout()

    if save_path:
        # transparent=False ensures white background is embedded in the SVG
        plt.savefig(save_path, format="svg", dpi=300, bbox_inches="tight", transparent=False)

    plt.show()

plot_trajectory(results_df.loc[7], model_conditions, save_path = "addm.svg")

In [ ]:
import pyddm

drift=0.4080
dt = 0.001

equivalent_model = pyddm.gddm(drift=drift, noise=0.235, T_dur=10, dx=0.001, dt=dt)
equivalent_model._overlay = pyddm.OverlayChain(overlays=[])
result = equivalent_model.simulate_trial(seed=42)

mpl.rcParams.update({
    "svg.fonttype": "none",
    "font.size": 18,        # 22 * 0.8
    "axes.titlesize": 21,   # 26 * 0.8 ≈ 21
    "axes.labelsize": 19,   # 24 * 0.8 ≈ 19
    "xtick.labelsize": 16,  # 20 * 0.8 ≈ 16
    "ytick.labelsize": 16,  # 20 * 0.8 ≈ 16
    "legend.fontsize": 16,  # 20 * 0.8 ≈ 16
})

def simple_plot_trajectory(x_positions, dt=dt, drift=drift, save_path=None):
    """
    Plot the decision variable trajectory with drift, boundaries, and reference lines.

    Parameters:
    - x_positions: list or np.array of decision variable values
    - dt: timestep size (default=0.01 seconds)
    - drift: drift rate (default=0.4714)
    """
    
    x_positions = np.array(x_positions)
    timesteps = np.arange(len(x_positions)) * dt

    fig, ax = plt.subplots(figsize=(12, 8))

    # Plot trajectory
    ax.plot(timesteps, x_positions, color="black", label="Trajectory")

    # Boundary lines (blue)
    ax.axhline(y=1, color="blue", linestyle="-", label="Boundary Separation")
    ax.axhline(y=-1, color="blue", linestyle="-", label="Boundary Separation")

    # Drift line (red dashed)
    drift_line = drift * timesteps
    ax.plot(timesteps, drift_line, color="red", linestyle="--", label=f"Drift (d={drift})")

    # Reference lines (black) at y=0 and x=0
    ax.axhline(y=0, color="black", linewidth=1)
    ax.axvline(x=0, color="black", linewidth=1)

    # X-axis scaling from -0.05 to 1.8
    ax.set_xlim(-0.05, 2.1)
    ax.set_ylim(-1.1, 1.1)

    # Labels and formatting
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Decision Variable")
    ax.set_title("Standard Decision Variable Trajectory")
    ax.grid(True)

    # Adjust legend to avoid duplicates
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc="lower right")

    if save_path:
        plt.savefig(save_path, format='svg', dpi=300)

    plt.tight_layout()
    plt.show()

simple_plot_trajectory(result, dt=equivalent_model.dt, save_path = "simple_ddm.svg")

This section contains the legacy model-free analysis pipeline that simulated data, formatted the simulated data, and plotted the data. `reformat_fixations` and `save_fixation_properties` and `save_basic_psychometrics` all take csv paths for the time being. Change to df's for hopper usage.